In [1]:
# importing libraries
import pandas as pd

In [2]:
# save filepath to variable for easier access
filepath = f"D:\Practice Projects\Python\Machine Learning\Kaggle Machine Learning\Melbourne Housing Problem\melb_data.csv"

melbourne_data = pd.read_csv(filepath)

# read the data and store data in DataFrame titled melbourne_data
melbourne_data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


#### Exploring the Missing Values in the Dataset
- number of columns with missing entries
- number of missing entries in each column
- total number of missing entries in the data

In [3]:
# shape of the data
melbourne_data.shape

(13580, 21)

In [ ]:
# number of missing values in each column
melbourne_data.isnull().sum()

Suburb              0
Address             0
Rooms               0
Type                0
Price               0
Method              0
SellerG             0
Date                0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

In [5]:
# columns with missing values
missing_val_count_by_column = melbourne_data.isnull().sum()


print(missing_val_count_by_column[missing_val_count_by_column > 0])

Car               62
BuildingArea    6450
YearBuilt       5375
CouncilArea     1369
dtype: int64


In [7]:
# number of columns with missing values
missing_val_count_by_column[missing_val_count_by_column > 0].shape[0]

4

In [8]:
# total number of missing values in the data
melbourne_data.isna().sum().sum()

np.int64(13256)

#### Assigning the variables, splitting the data --> training and validation sets


In [9]:
# splitting the data 
from sklearn.model_selection import train_test_split

In [10]:
# assigning target variables and dependent variables

y = melbourne_data.Price

melb_predictors = melbourne_data.drop(['Price'], axis=1)
X = melb_predictors.select_dtypes(exclude=['object'])

In [11]:
# dividing the data --> training = 0.8, testing = 0.2

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, 
                                                        test_size=0.2, random_state=0)

#### Define Function to Measure Quality of Each Approach
- Define a function called score_dataset() to evaluate model performance after each approach

In [ ]:
# loading the model
from sklearn.ensemble import RandomForestRegressor

# evaluation
from sklearn.metrics import mean_absolute_error

# Function for comparing different approaches
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=10, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

#### Score from Approach 1 (Drop Columns with Missing Values)

In [30]:
# Get names of columns with missing values
cols_with_missing = [col for col in X_train.columns if X_train[col].isnull().any()]
cols_with_missing


['Car', 'BuildingArea', 'YearBuilt']

In [15]:
# Drop columns in training and validation data
reduced_X_train = X_train.drop(cols_with_missing, axis=1)
reduced_X_valid = X_valid.drop(cols_with_missing, axis=1)



In [23]:
# show model performance
print("MAE from Approach 1 (Drop columns with missing values):", score_dataset(reduced_X_train, reduced_X_valid, y_train, y_valid))


MAE from Approach 1 (Drop columns with missing values): 183550.22137772635


#### Score from Approach 2 (Imputation)

In [17]:
# import the imputation library
from sklearn.impute import SimpleImputer

# Imputation
my_imputer = SimpleImputer()
imputed_X_train = pd.DataFrame(my_imputer.fit_transform(X_train))
imputed_X_valid = pd.DataFrame(my_imputer.transform(X_valid))

In [18]:
# Imputation removed column names; put them back
imputed_X_train.columns = X_train.columns
imputed_X_valid.columns = X_valid.columns

In [24]:
# evaluate the performance of the model
print("MAE from Approach 2 (Imputation):", score_dataset(imputed_X_train, imputed_X_valid, y_train, y_valid))


MAE from Approach 2 (Imputation): 178166.46269899711


#### Score from Approach 3 (An Extension to Imputation)
- Imputation and then further keeping track of the columns where changes occurred

In [25]:
# making a copy of the datasets
X_train_plus = X_train.copy()
X_valid_plus = X_valid.copy()

In [27]:
# Make new columns indicating what will be imputed
for col in cols_with_missing:
    X_train_plus[col + '_was_missing'] = X_train_plus[col].isnull()
    X_valid_plus[col + '_was_missing'] = X_valid_plus[col].isnull()

In [28]:
# Imputation
my_imputer = SimpleImputer()
imputed_X_train_plus = pd.DataFrame(my_imputer.fit_transform(X_train_plus))
imputed_X_valid_plus = pd.DataFrame(my_imputer.transform(X_valid_plus))

In [31]:
# Imputation removed column names; put them back
imputed_X_train_plus.columns = X_train_plus.columns
imputed_X_valid_plus.columns = X_valid_plus.columns

In [32]:
# evaluation model performance
print("MAE from Approach 3 (An Extension to Imputation):", score_dataset(imputed_X_train_plus, imputed_X_valid_plus, y_train, y_valid))


MAE from Approach 3 (An Extension to Imputation): 178927.503183954


---
- Imputation provides a better model performance compared to dropping the columns. 
- Only drop a column if the missing entries are more than 50% and the column doesn't contain relevant data